# Long-Term Aquatic Ecosystem Monitoring Observations and Social Media Data from the Basque Country, 1995–2015 Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://mlcroissant.org/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.mpbb-p57t/fair2.json`

This dataset includes structured observational and social media data collected over 20 years (1995–2015) from estuaries and coastal waters of the Basque Country, covering environmental monitoring from 51 stations. Variables comprise water, sediment, biota, phytoplankton, macroinvertebrates, fish, observer reports, and geolocated Twitter data, with quality assurance protocols and machine-readable metadata.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.mpbb-p57t/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")

# Optionally, show citation and version
print(f"Cite As: {metadata.cite_as}\nVersion: {metadata.version}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**All entities are referenced by their `@id`.**

In [ ]:
# List available record sets and their @id
record_sets = metadata.record_sets
print("Available RecordSets and their @id:")
for rs in record_sets:
    print(f"  - {rs.id}: {rs.name} ({rs.description if hasattr(rs, 'description') else ''})")

# For each record set, list its fields and their @id
for rs in record_sets:
    print(f"\nFields in RecordSet '{rs.name}' ({rs.id}):")
    for field in rs.fields:
        print(f"  - {field.id}: {field.name} ({getattr(field, 'description', '')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Select a RecordSet by its @id
# Replace with the record set @id you wish to analyze, for example:
example_record_set_id = None

for rs in record_sets:
    if example_record_set_id is None:
        example_record_set_id = rs.id

# Prepare dataframes for all record sets
dataframes = {}
for rs in record_sets:
    # Load all records for the record set
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df

if example_record_set_id in dataframes:
    print(f"\nColumns in RecordSet '{example_record_set_id}':")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All fields are referenced using their `@id`.

In [ ]:
# Example: Select a numeric field for analysis
# Find a numeric field @id from the earlier listing
numeric_field_id = None
group_field_id = None

# Identify numeric and group fields
rs = next((rs for rs in record_sets if rs.id == example_record_set_id), None)
for field in rs.fields:
    if getattr(field, 'data_type', None) in ['Integer', 'Float', 'Number'] and numeric_field_id is None:
        numeric_field_id = field.id
    if getattr(field, 'data_type', None) == 'Text' and group_field_id is None:
        group_field_id = field.id

# Show what fields have been chosen
print(f"Numeric field for EDA: {numeric_field_id}")
print(f"Group field for grouping: {group_field_id}")

df = dataframes[example_record_set_id]

if numeric_field_id in df.columns:
    # Filter out records with numeric_field > threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field (if exists)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        )
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure there's a numeric and group field for plotting
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

if numeric_field_id and group_field_id and\
   numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, you loaded a FAIR^2 dataset using the Croissant schema and explored environmental records spanning 20 years in the Basque Country. You:
- Inspected available record sets and their fields by @id.
- Loaded data into pandas DataFrames for analysis.
- Processed numeric variables, filtered and normalized values, and grouped by categorical attributes.
- Visualized distributions and relationships.

This approach enables FAIR data exploration and analysis with reproducible references to Croissant-defined entities. For further work, continue with domain-specific modeling or integrate environmental variables for predictive tasks.